# Nomis Mongo Sketches

Sketches on ingesting `nomis` JSON data directly into Mongo db and then processing it using Mongo queries.

The `nomis` database provides access to a wide range of data sets.

If we just want students to be able to access the data, we can point them to the `nomis` data download pages: https://www.nomisweb.co.uk/query/select/getdatasetbytheme.asp?opt=3&theme=&subgrp= The downloaded data can be downloaded in various formats (Excel, CSV, Nomis API format (i.e. JSON)).

But we can also provide a route to ingesting data into the Mongo DB databases via a Python API and the nomis web API service.

## Mongo Connection

Typically, we might set up a Mongo DB connection as follows:

In [ ]:
from pymongo import MongoClient

MONGO_CONNECTION_STRING = f"mongodb://localhost:27017/"
mongo_client = MongoClient(MONGO_CONNECTION_STRING)

DB_NAME = "nomis_test_db"
mongo_db = mongo_client[DB_NAME]

datasets_index_collection = mongo_db["datasets_index"]

## Download directory of `nomis` datasets

Start off by getting the list of available datasets:

In [ ]:
import requests
import urllib
import pandas as pd

"""
To keep eveything together, we can bundle a set of functions
in a package or a class. Or both.

Let's start with a stub of a class.
"""


class NOMIS_MONGO:
    def __init__(self):
        NOMIS_STUB = "https://www.nomisweb.co.uk/api/v01/dataset/"

        self.url = NOMIS_STUB
        self.codes = None
        self.metadata = {}

    def _url_encode(self, params=None):
        if params is not None and params != "" and params != {}:
            # params='?{}'.format( '&'.join( ['{}={}'.format(p,params[p]) for p in params] ) )
            params = "?{}".format(urllib.parse.urlencode(params))
        else:
            params = ""
        return params

    def _get_datasets(self, search=None):
        url = f"{self.url}def.sdmx.json"
        if search is not None:
            url = "{url}{params}".format(
                url=url, params=self._url_encode({"search": search})
            )
        r = requests.get(url)
        # By instpection of the response,
        # let's navigate to the list or
        return r.json()["structure"]["keyfamilies"]["keyfamily"]

In [ ]:
# Instantiate the class
nomis = NOMIS_MONGO()

In [ ]:
# Get the index records of available datasets in nomis
# and insert them into a Mongo collection
_ = datasets_index_collection.insert_many(nomis._get_datasets())

# I suppose we could extend the NOMIS_MONGO class to set up
# a database connection and index collection and provide a method to query it

In [ ]:
# Review a single record
datasets_index_collection.find_one()

{'_id': ObjectId('683990c12a984c141b4ffbc4'),
 'agencyid': 'NOMIS',
 'annotations': {'annotation': [{'annotationtext': 'Current (being actively updated)',
    'annotationtitle': 'Status'},
   {'annotationtext': 'Claimants,JSA,Rates', 'annotationtitle': 'Keywords'},
   {'annotationtext': 'Persons', 'annotationtitle': 'Units'},
   {'annotationtext': 'jsa', 'annotationtitle': 'contenttype/sources'},
   {'annotationtext': 1, 'annotationtitle': 'contenttype/sorting'},
   {'annotationtext': 'usw91 usp usu', 'annotationtitle': 'SubDescription'},
   {'annotationtext': 'us', 'annotationtitle': 'Mnemonic'},
   {'annotationtext': '2004-06-16 09:30:00',
    'annotationtitle': 'FirstReleased'},
   {'annotationtext': '2025-05-13 07:00:00',
    'annotationtitle': 'LastUpdated'}]},
 'id': 'NM_1_1',
 'components': {'attribute': [{'assignmentstatus': 'Mandatory',
    'attachmentlevel': 'Observation',
    'codelist': 'CL_OBS_STATUS',
    'conceptref': 'OBS_STATUS'},
   {'assignmentstatus': 'Conditional',

The `annotations` provide useful metadata about each data souce, but in an "exploded" way, of the form `{"key": key, "value": value}` rather than `{key:value}`.

The following pipeline *(originally suggested by Claude.ai based on viewing a single provided record)* originally parsed the annotations to search for annotation declared `contenttype/sources`. The function was then refactored to call a more general annotation parsing function, which uses an `$unwind` operator as well as a `$group` operator. The `$unwind` operator appears in notebook 15.3; the `$group` operator is introduced in notebook 15.4.

```{admonition} MT Note
:class: dange

Notebooks for week 16 have been dropped over the years, so there may be an opportunity to add one back and use this dataset as part of a worked example using "found" data to populate and work with a Mongo database?
```

In [ ]:
# These base funciton at least is an obvious candidate
# for bundling back into the base nomis class...


def _nomis_parse_annotations(field):
    pipeline = [
        {"$unwind": "$annotations.annotation"},
        {"$match": {"annotations.annotation.annotationtitle": field}},
        {
            "$group": {"_id": "$annotations.annotation.annotationtext"}
        },  # We want just the unique values
        {"$sort": {"_id": 1}},
        {"$project": {"_id": 0, "source": "$_id"}},
    ]
    return [r["source"] for r in datasets_index_collection.aggregate(pipeline)]


def nomis_data_sources():
    """Get a list of nomis data sources from the nomis index records collection."""
    return _nomis_parse_annotations("contenttype/sources")


nomis_data_sources()

['acses',
 'aps',
 'ashe',
 'bres',
 'cc',
 'census_1921_cr',
 'census_1961_sh',
 'census_1961_st',
 'census_1981',
 'census_1991',
 'census_2001',
 'census_2001_cs',
 'census_2001_ct',
 'census_2001_ks',
 'census_2001_st',
 'census_2001_tt',
 'census_2001_uv',
 'census_2011_ct',
 'census_2011_dc',
 'census_2011_dc,census_2011_dc',
 'census_2011_ks',
 'census_2011_ksuk',
 'census_2011_lc',
 'census_2011_od',
 'census_2011_ot',
 'census_2011_qs',
 'census_2011_qsuk',
 'census_2011_st',
 'census_2011_ukmigration',
 'census_2011_wd',
 'census_2011_wp',
 'census_2021_asg',
 'census_2021_pp',
 'census_2021_rm',
 'census_2021_st',
 'census_2021_ts',
 'dwp_5pc',
 'dwp_wpls',
 'housing',
 'jd',
 'jsa',
 'life',
 'pest',
 'ra',
 'ukbc',
 'vacs',
 'vat',
 'wfj']

How easy is it to search the collection?

Find records where the source is census 2021 flavour:

In [51]:
import re

census_2021_docs = datasets_index_collection.find(
    {
        "annotations.annotation": {
            "$elemMatch": {
                "annotationtitle": "contenttype/sources",
                # The regex captures the different flavours of census_2021 data
                "annotationtext": {
                    "$regex": re.compile("^census_2021.*", re.IGNORECASE)
                },
            }
        }
    }
)

# The interesting columns are id and name.value
# pd.json_normalize(census_2021_docs)
pd.json_normalize(census_2021_docs)[["id", "name.value"]]

,id,name.value
0,NM_2020_1,TS007A - Age by five-year age bands
1,NM_2021_1,TS001 - Number of usual residents in household...
2,NM_2022_1,TS002 - Legal partnership status
3,NM_2023_1,TS003 - Household composition
4,NM_2024_1,TS004 - Country of birth
...,...,...
315,NM_2360_1,PP010 - Car or van availability
316,NM_2361_1,PP011 - Occupation
317,NM_2362_1,PP012 - Age
318,NM_2363_1,PP013 - Travel to work


```{admonition} MT Note
:class: danger

It would be nice if students could develop their own packages or classes
(which is to say, their own packaged toolsets) through the teaching weeks,
e.g. for accessing (via APIs,. bulk download URls etc.), cleaning and
parsing data sets etc.

If we defined a series of tests, students could develop their own tools to 
satisfy the tests. We could also provide a minimal package that satisifies
the same tests, which could be used as a fallback in assessment if a student
failed to create their own package or wanted the assurance of using an implemented
package that worked as the MT expected to support the assessment activity.
```

## Fetching a census dataset and loading it into Mongo

Create a simple function to grab a dataset:

In [ ]:
def get_dataset(idx, geography=None):
    params = {}
    if geography is not None:
        params["geography"] = geography

    url = "{nomis}{idx}.data.json{params}".format(
        nomis=nomis.url, idx=idx, params=nomis._url_encode(params)
    )
    # print(url)
    r = requests.get(url)
    return r.json()["obs"]


# The geography code is one I had already discovered: LSOAs in Isle of Wight LAD
# Isle of Wight LAD: 645923034
# In this dataset, LSOAs: TYPE151

# For assessment purposes, we could provide these codes
# "download THIS dataset (`NM_xxx`) for THIS geopgraphy (`MMMMTYPENNN`)"
# and also provide that (or enough of a fragment of it for the assessment)
# in a data directory.
resident_data = get_dataset("NM_2021_1", "645923034TYPE151")
# resident_data

[{'dataset': {'value': 'NM_2021_1',
   'description': 'TS001 - Number of usual residents in households and communal establishments'},
  'geography': {'value': 633356334,
   'description': 'Isle of Wight 001A',
   'geogcode': 'E01017300'},
  'c2021_restype_3': {'value': 0, 'description': 'Total: All usual residents'},
  'measures': {'value': 20100, 'description': 'Value'},
  'freq': 'A',
  'time_format': {'value': 'P1Y', 'description': 'Annual'},
  'unit': {'value': 'PER', 'description': 'Persons'},
  'time': {'value': 2021, 'description': 2021},
  'obs_value': {'value': 2151,
   'description': 'Persons Value for Isle of Wight 001A (geography) for 2021 from TS001 - Number of usual residents in households and communal establishments'},
  'obs_status': {'value': 'A', 'description': 'Normal Value'},
  'obs_conf': {'value': 'F', 'description': 'Free (free for publication)'},
  'obs_round': {'value': 0, 'description': 'Not rounded'},
  'urn': 'Nm-2021d1d32336e0d633356334d0d20100'},
 {'datase

Create a collection for the dataset:

In [ ]:
# Create a new collection
iw_lsoa_collection = mongo_db["iw_lsoa"]

# Add the data to it
_ = iw_lsoa_collection.insert_many(resident_data)

View a sample record:

In [ ]:
# Review an example record
iw_lsoa_collection.find_one()

{'_id': ObjectId('6839b7e12a984c141b500213'),
 'dataset': {'value': 'NM_2021_1',
  'description': 'TS001 - Number of usual residents in households and communal establishments'},
 'geography': {'value': 633356334,
  'description': 'Isle of Wight 001A',
  'geogcode': 'E01017300'},
 'c2021_restype_3': {'value': 0, 'description': 'Total: All usual residents'},
 'measures': {'value': 20100, 'description': 'Value'},
 'freq': 'A',
 'time_format': {'value': 'P1Y', 'description': 'Annual'},
 'unit': {'value': 'PER', 'description': 'Persons'},
 'time': {'value': 2021, 'description': 2021},
 'obs_value': {'value': 2151,
  'description': 'Persons Value for Isle of Wight 001A (geography) for 2021 from TS001 - Number of usual residents in households and communal establishments'},
 'obs_status': {'value': 'A', 'description': 'Normal Value'},
 'obs_conf': {'value': 'F', 'description': 'Free (free for publication)'},
 'obs_round': {'value': 0, 'description': 'Not rounded'},
 'urn': 'Nm-2021d1d32336e0d6

In [60]:
# Add some hours worked data
_ = iw_lsoa_collection.insert_many(get_dataset("NM_2076_1", "645923034TYPE151"))

In [ ]:
# Review one of those records
iw_lsoa_collection.find_one({"dataset.value": "NM_2076_1"})

{'_id': ObjectId('6839b8a62a984c141b500429'),
 'dataset': {'value': 'NM_2076_1', 'description': 'TS059 - Hours worked'},
 'geography': {'value': 633356334,
  'description': 'Isle of Wight 001A',
  'geogcode': 'E01017300'},
 'c2021_hours_5': {'value': 0,
  'description': 'Total: All usual residents aged 16 years and over in employment the week before the census'},
 'measures': {'value': 20100, 'description': 'Value'},
 'freq': 'A',
 'time_format': {'value': 'P1Y', 'description': 'Annual'},
 'unit': {'value': 'PER', 'description': 'Persons'},
 'time': {'value': 2021, 'description': 2021},
 'obs_value': {'value': 925,
  'description': 'Persons Value for Isle of Wight 001A (geography) for 2021 from TS059 - Hours worked'},
 'obs_status': {'value': 'A', 'description': 'Normal Value'},
 'obs_conf': {'value': 'F', 'description': 'Free (free for publication)'},
 'obs_round': {'value': 0, 'description': 'Not rounded'},
 'urn': 'Nm-2076d1d32336e0d633356334d0d20100'}

The previous code cell perhaps naturally suggests the following function:

In [ ]:
def preview_dataset_record(collection, dataset):
    return collection.find_one({"dataset.value": dataset})


preview_dataset_record(iw_lsoa_collection, "NM_2076_1")

# Alternatively, we could add it as a method to the collection
# either by monkey patching or by creating iw_lsoa_collection
# as an extended class, e.g.
"""
from pymongo.collection import Collection

class NomisDatasetCollection(Collection):
    def preview_dataset_record(self, dataset):
        return self.find_one({"dataset.value": dataset})

iw_lsoa_collection2 = NomisDatasetCollection(mongo_db, "iw_lsoa2")
_ = iw_lsoa_collection2.insert_many(get_dataset("NM_2076_1", "645923034TYPE151"))
iw_lsoa_collection2.preview_dataset_record("NM_2076_1")
"""

{'_id': ObjectId('6839b8a62a984c141b500429'),
 'dataset': {'value': 'NM_2076_1', 'description': 'TS059 - Hours worked'},
 'geography': {'value': 633356334,
  'description': 'Isle of Wight 001A',
  'geogcode': 'E01017300'},
 'c2021_hours_5': {'value': 0,
  'description': 'Total: All usual residents aged 16 years and over in employment the week before the census'},
 'measures': {'value': 20100, 'description': 'Value'},
 'freq': 'A',
 'time_format': {'value': 'P1Y', 'description': 'Annual'},
 'unit': {'value': 'PER', 'description': 'Persons'},
 'time': {'value': 2021, 'description': 2021},
 'obs_value': {'value': 925,
  'description': 'Persons Value for Isle of Wight 001A (geography) for 2021 from TS059 - Hours worked'},
 'obs_status': {'value': 'A', 'description': 'Normal Value'},
 'obs_conf': {'value': 'F', 'description': 'Free (free for publication)'},
 'obs_round': {'value': 0, 'description': 'Not rounded'},
 'urn': 'Nm-2076d1d32336e0d633356334d0d20100'}

In [ ]:
# Maybe even the following:
import pymongo


def get_and_add_dataset(collection, dataset, geography="", mongo_db=None, preview=True):
    if isinstance(collection, pymongo.collection.Collection):
        pass
    elif collection and isinstance(collection, str) and mongo_db:
        collection = mongo_db[collection]
    else:
        return

    data = get_dataset(dataset, geography)

    # Add the data to it
    _ = iw_lsoa_collection.insert_many(data)

    if preview:
        return preview_dataset_record(collection, dataset)

    return

## Identifying Geography Codes

Downloading data from the APi for a particular gepgraphy reduces the amount of data requested and the amopunt we have to store. An alternative would be for students to pull all the data and then search within it; but that may cause resource issues for some students, and can really hammer the ONS API. There may also be a maximum record limit (I think tjhat can be increased by getting a free API key).

The problem is, finding the geography code can be a little fiddly...

The following function attempts to simplify the process using the `nomis` API geography lookup service:

In [ ]:
def nomis_codes_geog(idx, geography=None, search=None):
    params = {}
    if geography is not None:
        geog = "/{geog}".format(geog=geography)
    else:
        geog = ""

    if search is not None:
        params["search"] = search

    url = "{nomis}{idx}/geography{geog}.def.sdmx.json{params}".format(
        nomis=nomis.url, idx=idx, geog=geog, params=nomis._url_encode(params)
    )
    # return requests.get(url).json()
    return requests.get(url).json()["structure"]["codelists"]["codelist"][0]["code"]

By default, it looks up the top level geographies associated with a particular dataset.

In [89]:
nomis_codes_geog("NM_2076_1")

[{'annotations': {'annotation': [{'annotationtext': 'countries',
     'annotationtitle': 'TypeName'},
    {'annotationtext': 499, 'annotationtitle': 'TypeCode'},
    {'annotationtext': 'K04000001', 'annotationtitle': 'GeogCode'}]},
  'description': {'value': 'England and Wales', 'lang': 'en'},
  'value': 2092957703},
 {'annotations': {'annotation': [{'annotationtext': 'countries',
     'annotationtitle': 'TypeName'},
    {'annotationtext': 499, 'annotationtitle': 'TypeCode'},
    {'annotationtext': 'E92000001', 'annotationtitle': 'GeogCode'}]},
  'description': {'value': 'England', 'lang': 'en'},
  'value': 2092957699},
 {'annotations': {'annotation': [{'annotationtext': 'countries',
     'annotationtitle': 'TypeName'},
    {'annotationtext': 499, 'annotationtitle': 'TypeCode'},
    {'annotationtext': 'W92000004', 'annotationtitle': 'GeogCode'}]},
  'parentcode': 2092957700,
  'description': {'value': 'Wales', 'lang': 'en'},
  'value': 2092957700}]

Use the `England and Wales` geography code (the *value* `2092957703`):

In [95]:
geogs_json = nomis_codes_geog("NM_2076_1", geography=2092957703)
geogs_json

[{'annotations': {'annotation': [{'annotationtext': 'countries',
     'annotationtitle': 'TypeName'},
    {'annotationtext': 499, 'annotationtitle': 'TypeCode'},
    {'annotationtext': 'K04000001', 'annotationtitle': 'GeogCode'}]},
  'description': {'value': 'England and Wales', 'lang': 'en'},
  'value': 2092957703},
 {'annotations': {'annotation': [{'annotationtext': 'True',
     'annotationtitle': 'IsAbstractCode'},
    {'annotationtext': 150, 'annotationtitle': 'TypeCode'},
    {'annotationtext': '2021 output areas', 'annotationtitle': 'TypeName'},
    {'annotationtext': 2092957703, 'annotationtitle': 'ParentCode'},
    {'annotationtext': 188880, 'annotationtitle': 'ChildCount'}]},
  'description': {'value': '2021 output areas within England and Wales',
   'lang': 'en'},
  'value': '2092957703TYPE150'},
 {'annotations': {'annotation': [{'annotationtext': 'True',
     'annotationtitle': 'IsAbstractCode'},
    {'annotationtext': 151, 'annotationtitle': 'TypeCode'},
    {'annotationtex

In [ ]:
# By inspections, we can parse the response into a pandas dataframe
# (alternatively we could load it in toa Mondo DB collection and query it)
geogs = []
for r in geogs_json:
    geogs.append((r["description"]["value"], r["value"]))

pd.DataFrame(geogs, columns=["name", "geography"])

# We could also bundle the above into our nomis_codes_geog function
# perhaps with an option of: return_df=bool

,name,geography
0,England and Wales,2092957703
1,2021 output areas within England and Wales,2092957703TYPE150
2,2021 super output areas - lower layer within E...,2092957703TYPE151
3,2021 super output areas - middle layer within ...,2092957703TYPE152
4,2022 wards within England and Wales,2092957703TYPE153
5,2022 local authorities: districts within Engla...,2092957703TYPE154
6,2022 local authorities: counties within Englan...,2092957703TYPE155
7,2021 national parks within England and Wales,2092957703TYPE168
8,local authorities: county / unitary (as of Apr...,2092957703TYPE423
9,local authorities: district / unitary (as of A...,2092957703TYPE424


In [ ]:
# Use the lower layer (LSOA) geography code for England
iw_lsoa_json = nomis_codes_geog(
    "NM_2076_1", geography="2092957703TYPE151", search="Isle of Wight*"
)
iw_lsoa_json

[{'annotations': {'annotation': [{'annotationtext': '2021 super output areas - lower layer',
     'annotationtitle': 'TypeName'},
    {'annotationtext': 151, 'annotationtitle': 'TypeCode'},
    {'annotationtext': 'E01017300', 'annotationtitle': 'GeogCode'}]},
  'description': {'value': 'Isle of Wight 001A', 'lang': 'en'},
  'value': 633356334},
 {'annotations': {'annotation': [{'annotationtext': '2021 super output areas - lower layer',
     'annotationtitle': 'TypeName'},
    {'annotationtext': 151, 'annotationtitle': 'TypeCode'},
    {'annotationtext': 'E01017303', 'annotationtitle': 'GeogCode'}]},
  'description': {'value': 'Isle of Wight 001B', 'lang': 'en'},
  'value': 633356337},
 {'annotations': {'annotation': [{'annotationtext': '2021 super output areas - lower layer',
     'annotationtitle': 'TypeName'},
    {'annotationtext': 151, 'annotationtitle': 'TypeCode'},
    {'annotationtext': 'E01017305', 'annotationtitle': 'GeogCode'}]},
  'description': {'value': 'Isle of Wight 001C

In [124]:
# Get the list of IW LSOA codes
iw_lsoa_codes = [r["value"] for r in iw_lsoa_json]

## Finding Shapefile Codes for Local Authority Districts

The LAD/LSOA shapefiles that can be accessed from https://github.com/drkane/geo-lookups/ are keyed using Local Authority District codes of the form `E06000046`.

For the `NM_2021_1` dataset we see the following top level geographies:

In [ ]:
def get_geogs_df(geogs_json, chase=False):
    geogs = []
    for r in geogs_json:
        geogs.append((r["description"]["value"], r["value"]))

    if not chase:
        return pd.DataFrame(geogs, columns=["name", "geography"])


geogs_df = get_geogs_df(nomis_codes_geog("NM_2021_1"))
geogs_df

,name,geography
0,England and Wales,2092957703
1,England,2092957699
2,Wales,2092957700


In [141]:
# Tunnel down into England and Wales...
get_geogs_df(
    nomis_codes_geog(
        "NM_2021_1",
        geography=geogs_df[geogs_df["name"] == "England and Wales"]["geography"].iloc[
            0
        ],
    )
)

,name,geography
0,England and Wales,2092957703
1,2021 output areas within England and Wales,2092957703TYPE150
2,2021 super output areas - lower layer within E...,2092957703TYPE151
3,2021 super output areas - middle layer within ...,2092957703TYPE152
4,2022 wards within England and Wales,2092957703TYPE153
5,2022 local authorities: districts within Engla...,2092957703TYPE154
6,2022 local authorities: counties within Englan...,2092957703TYPE155
7,2021 Parliamentary constituencies within Engla...,2092957703TYPE158
8,2021 national parks within England and Wales,2092957703TYPE168
9,Westminster Parliamentary Constituencies (July...,2092957703TYPE172


We see there is a geography associated with *2022 local authorities: districts within England and Wales*:

In [ ]:
# Search LADs:
# From the above: 2022 local authorities: districts within England and Wales
# which has gepography code: 2092957703TYPE154

# We can now search for the LAD code for the Isle of Wight
iw_geog = nomis_codes_geog(
    "NM_2021_1", geography="2092957703TYPE154", search="Isle of Wight*"
)
iw_geog
# This data includes a LAD identifier given as the GeogCode: E06000046

[{'annotations': {'annotation': [{'annotationtext': '2022 local authorities: districts',
     'annotationtitle': 'TypeName'},
    {'annotationtext': 154, 'annotationtitle': 'TypeCode'},
    {'annotationtext': 'E06000046', 'annotationtitle': 'GeogCode'}]},
  'parentcode': 654311432,
  'description': {'value': 'Isle of Wight', 'lang': 'en'},
  'value': 645923034}]

Let's create a helper to parse the JSON / dict in Python (rather than querying this in Mongo, for example):

In [153]:
def get_annotation_from_annotations(data, field):
    return next(
        (
            annotation["annotationtext"]
            for annotation in data["annotations"]["annotation"]
            if annotation["annotationtitle"] == field
        ),
        None,  # default value if not found
    )


iw_LAD_code = get_annotation_from_annotations(iw_geog[0], "GeogCode")
iw_LAD_code

'E06000046'

In [ ]:
import folium

# From previous example in another notebook


def get_drkane_geoboundary(id, typ="LSOA"):
    """Start to create a function to pull data from drkane's geo repo."""
    base_url = "https://raw.githubusercontent.com/drkane/geo-lookups/refs/heads/master/boundaries/"
    url = ""
    if typ == "LSOA":
        url = f"{base_url}lsoa/{id}.geojson"
    if url:
        return requests.get(url).json()


# Example of getting the LSOA boundaries within an LA
gj_data = get_drkane_geoboundary(iw_LAD_code)

m = folium.Map()
geo_j = folium.GeoJson(data=gj_data)
geo_j.add_to(m)

# auto zoom in...
m.fit_bounds(m.get_bounds())
m

In [158]:
# Bundling further...


# We want to find all the geographies, perhaps
def nomis_codes_geog_all(idx):
    top_level = get_geogs_df(nomis_codes_geog(idx))
    # TO DO - we need ensure there is a proper df, even if empty
    if not top_level.empty:
        for geog in top_level["geography"]:
            _df = get_geogs_df(nomis_codes_geog(idx, geog))
            top_level = pd.concat([top_level, _df])
    return top_level.drop_duplicates().reset_index()


nomis_codes_geog_all("NM_2021_1")

,index,name,geography
0,0,England and Wales,2092957703
1,1,England,2092957699
2,2,Wales,2092957700
3,1,2021 output areas within England and Wales,2092957703TYPE150
4,2,2021 super output areas - lower layer within E...,2092957703TYPE151
5,3,2021 super output areas - middle layer within ...,2092957703TYPE152
6,4,2022 wards within England and Wales,2092957703TYPE153
7,5,2022 local authorities: districts within Engla...,2092957703TYPE154
8,6,2022 local authorities: counties within Englan...,2092957703TYPE155
9,7,2021 Parliamentary constituencies within Engla...,2092957703TYPE158


## Accessing other Census datasets

Let's look at one of the other datasets that I think are the ones specified in the TM351 25J EMA draft:

In [125]:
# Occupation data: NM_2080_1
# We can use the geography code list in a query
_ = iw_lsoa_collection.insert_many(get_dataset("NM_2080_1", iw_lsoa_codes))

In [114]:
iw_lsoa_collection.find_one({"dataset.value": "NM_2080_1"})

{'_id': ObjectId('6839c2d92a984c141b500907'),
 'dataset': {'value': 'NM_2080_1', 'description': 'TS063 - Occupation'},
 'geography': {'value': 633356334,
  'description': 'Isle of Wight 001A',
  'geogcode': 'E01017300'},
 'c2021_occ_10': {'value': 0,
  'description': 'Total: All usual residents aged 16 years and over in employment the week before the census'},
 'measures': {'value': 20100, 'description': 'Value'},
 'freq': 'A',
 'time_format': {'value': 'P1Y', 'description': 'Annual'},
 'unit': {'value': 'PER', 'description': 'Persons'},
 'time': {'value': 2021, 'description': 2021},
 'obs_value': {'value': 930,
  'description': 'Persons Value for Isle of Wight 001A (geography) for 2021 from TS063 - Occupation'},
 'obs_status': {'value': 'A', 'description': 'Normal Value'},
 'obs_conf': {'value': 'F', 'description': 'Free (free for publication)'},
 'obs_round': {'value': 0, 'description': 'Not rounded'},
 'urn': 'Nm-2080d1d32336e0d633356334d0d20100'}

In [116]:
# Find the percentage value
iw_lsoa_collection.find_one(
    {"dataset.value": "NM_2080_1", "measures.description": "Percent"}
)

{'_id': ObjectId('6839c2d92a984c141b500911'),
 'dataset': {'value': 'NM_2080_1', 'description': 'TS063 - Occupation'},
 'geography': {'value': 633356334,
  'description': 'Isle of Wight 001A',
  'geogcode': 'E01017300'},
 'c2021_occ_10': {'value': 0,
  'description': 'Total: All usual residents aged 16 years and over in employment the week before the census'},
 'measures': {'value': 20301, 'description': 'Percent'},
 'freq': 'A',
 'time_format': {'value': 'P1Y', 'description': 'Annual'},
 'unit': {'value': 'PER', 'description': 'Persons'},
 'time': {'value': 2021, 'description': 2021},
 'obs_value': {'value': 100.0,
  'description': 'Persons Percent for Isle of Wight 001A (geography) for 2021 from TS063 - Occupation'},
 'obs_status': {'value': 'A', 'description': 'Normal Value'},
 'obs_conf': {'value': 'F', 'description': 'Free (free for publication)'},
 'urn': 'Nm-2080d1d32336e0d633356334d0d20301'}

In [ ]:
# We can hand roll getting a pandas df from a Mongo DB query:
pd.json_normalize(
    iw_lsoa_collection.find(
        {"dataset.value": "NM_2080_1", "measures.description": "Value"},
        {
            "description": 1,
            "geography.description": 1,
            "geography.geogcode": 1,
            "obs_value.value": 1,
            "c2021_occ_10.description": 1,
            "_id": 0,
        },
    )
)

,geography.description,geography.geogcode,c2021_occ_10.description,obs_value.value
0,Isle of Wight 001A,E01017300,Total: All usual residents aged 16 years and o...,930
1,Isle of Wight 001A,E01017300,"1. Managers, directors and senior officials",201
2,Isle of Wight 001A,E01017300,2. Professional occupations,216
3,Isle of Wight 001A,E01017300,3. Associate professional and technical occupa...,117
4,Isle of Wight 001A,E01017300,4. Administrative and secretarial occupations,59
...,...,...,...,...
885,Isle of Wight 018E,E01017370,5. Skilled trades occupations,87
886,Isle of Wight 018E,E01017370,"6. Caring, leisure and other service occupations",85
887,Isle of Wight 018E,E01017370,7. Sales and customer service occupations,73
888,Isle of Wight 018E,E01017370,"8. Process, plant and machine operatives",51
